# MLOps Workflow for Model Deployment in Client Projects


In this walkthrough, you will manage an ML workflow for deploying a scalable predictive model for a financial client.

**Scenario:**
Your financial services client wants a fraud detection model that can scale across multiple regions with minimal downtime. Your team is responsible for deploying the model and ensuring version contro

**Objective:**
- Preprocess the dataset for model training and train the model
- Use MLflow to track experiments, hyperparameters, and model versions.

### Part 1: Load the 'fraud_detection_dataset.csv' from the data directory into a Pandas DataFrame.
**Task 1:**
- Load and Preprocess the dataset.
- Split the dataset into training and testing datasets

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("data/fraud_detection_dataset.csv")

# 'type' (TRANSFER, CASH_IN...) to 0-1
type_dummies = pd.get_dummies(data['type'], drop_first=True)
data_new = pd.concat([data, type_dummies], axis=1)

X = data_new.drop(['isFraud', 'type', 'nameOrg', 'nameDest'], axis=1)
y = data_new['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(data.shape)
print(y.value_counts())
data.head()


(20020, 10)
0    19609
1      411
Name: isFraud, dtype: int64


,step,type,amount,nameOrg,oldbalanceOrg,newbalanceOrg,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,103,TRANSFER,4215.77,C30199,9962.31,5746.54,C7645,18118.40,18118.40,0
1,436,CASH_IN,2805.48,C8699,16773.83,13968.35,C27141,13150.58,15956.06,0
2,861,TRANSFER,8950.44,C26191,15865.27,6914.83,C20483,10882.61,10882.61,0
3,271,DEBIT,3322.40,C973,8259.44,4937.04,C7632,8104.84,11427.24,0
4,107,CASH_IN,5785.97,C13679,19524.09,13738.12,C35689,5044.86,10830.83,0


### Part 2: Start a Local MLflow Server

**Task:**
- Install and import mlflow 
- Set the Tracking Server URI
- Train a model and prepare metadata for logging
- Start an MLflow run

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Fraud Detection")


def train_and_log_model(model, model_name):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions)
        recall = recall_score(y_test, predictions)
        f1 = f1_score(y_test, predictions)
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

        mlflow.log_param("model_name", model_name)
        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("auc", auc)
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"\n{model_name}")
        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1:        {f1:.4f}")
        print(f"AUC:       {auc:.4f}")
        print(confusion_matrix(y_test, predictions))

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)

train_and_log_model(lr_model, "LogisticRegression_Balanced")
train_and_log_model(rf_model, "RandomForest_Balanced")


2026/10/22 20:05:11 INFO mlflow.tracking.fluent: Experiment with name 'Fraud Detection' does not exist. Creating a new experiment.



LogisticRegression_Balanced
Accuracy:  0.5386
Precision: 0.0210
Recall:    0.4496
F1:        0.0402
AUC:       0.5076
[[3177 2700]
 [  71   58]]

RandomForest_Balanced
Accuracy:  0.8705
Precision: 0.0193
Recall:    0.1008
F1:        0.0323
AUC:       0.4608
[[5215  662]
 [ 116   13]]


/opt/venv/lib/python3.11/site-packages/_distutils_hack/__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/opt/venv/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


### Part 3:  View the Run in the MLflow UI

**Task:**
- Using Tracking Server at http://localhost:5000, start the MLflow UI
- Click on the name of the Experiment that is created (“Fraud Detection”) to see the list of runs associated with the Experiment.

In [3]:
runs = mlflow.search_runs(experiment_names=["Fraud Detection"])
runs[["tags.mlflow.runName", "metrics.accuracy", "metrics.precision",
      "metrics.recall", "metrics.f1_score", "metrics.auc"]]


,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.auc
0,RandomForest_Balanced,0.870463,0.019259,0.100775,0.032338,0.460787
1,LogisticRegression_Balanced,0.538628,0.021030,0.449612,0.040180,0.507594
